<a href="https://colab.research.google.com/github/vadim13213/neural_networks/blob/main/new_2026_8.%20%D0%93%D0%B5%D0%BD%D0%B5%D1%80%D0%B0%D1%82%D0%B8%D0%B2%D0%BD%D0%BE-%D1%81%D0%BE%D1%81%D1%82%D1%8F%D0%B7%D0%B0%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D0%B0%D1%8F%20%D1%81%D0%B5%D1%82%D1%8C%20(GAN)/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%968_%D0%93%D0%B5%D0%BD%D0%B5%D1%80%D0%B0%D1%82%D0%B8%D0%B2%D0%BD%D0%BE_%D1%81%D0%BE%D1%81%D1%82%D1%8F%D0%B7%D0%B0%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D0%B0%D1%8F_%D1%81%D0%B5%D1%82%D1%8C_(GAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа №8. Генеративно-состязательная сеть (GAN)**

---
**❗ Примечание:**

Не забывайте периодически сохранять параметры модели. Функции для этого описаны в теоретической части. В случае приостановки процесса обучения из-за перегрузки ОЗУ, Вы сможете загрузить последние предобученные параметры и продолжить обучение.

---

## **Задание №1.** Обучите генератор воспризводить примитивные изображения. Датасет выберите по желанию. ([Пример №1](https://www.kaggle.com/datasets/andrewmvd/medical-mnist), [Пример №2](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/fashion_mnist/load_data#example), [Пример №3](https://www.kaggle.com/datasets/sagyamthapa/handwritten-math-symbols))





In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

import numpy as np
import matplotlib.pyplot as plt

import os
import random

from IPython import display

In [ ]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
os.makedirs("gan_generated_images", exist_ok=True)
os.makedirs("gan_saved_models", exist_ok=True)

In [ ]:
# Загрузка Fashion-MNIST
(x_train, _), (_, _) = tf.keras.datasets.fashion_mnist.load_data()

# Нормализация в диапазон [-1, 1]
x_train = x_train.astype("float32") / 127.5 - 1

# Добавляем канал
x_train = np.expand_dims(x_train, axis=-1)

print(x_train.shape)

Должно быть (60000, 28, 28, 1)

In [ ]:
noise_dim = 100


def build_generator():
    model = tf.keras.Sequential([
        layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(noise_dim,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Reshape((7, 7, 256)),

        layers.Conv2DTranspose(128, 5, strides=1, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(64, 5, strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(
            1,
            5,
            strides=2,
            padding='same',
            use_bias=False,
            activation='tanh'
        )
    ])

    return model

In [ ]:
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Conv2D(64, 5, strides=2, padding='same', input_shape=[28, 28, 1]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(128, 5, strides=2, padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1)
    ])

    return model

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)


def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss



def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

In [ ]:
generator = build_generator()
discriminator = build_discriminator()


generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)


generator.summary()
discriminator.summary()

In [ ]:
BATCH_SIZE = 256
BUFFER_SIZE = 60000
EPOCHS = 50

train_dataset = (
    tf.data.Dataset
    .from_tensor_slices(x_train)
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
)

In [ ]:
@tf.function
def train_step(images):

    noise = tf.random.normal([BATCH_SIZE, noise_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:

        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    gradients_of_discriminator = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    generator_optimizer.apply_gradients(
        zip(gradients_of_generator, generator.trainable_variables)
    )

    discriminator_optimizer.apply_gradients(
        zip(gradients_of_discriminator, discriminator.trainable_variables)
    )

In [ ]:
num_examples_to_generate = 16
seed = tf.random.normal([num_examples_to_generate, noise_dim])


def generate_and_save_images(model, epoch, test_input):

    predictions = model(test_input, training=False)

    fig = plt.figure(figsize=(4, 4))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow((predictions[i, :, :, 0] + 1) / 2.0, cmap='gray')
        plt.axis('off')

    plt.savefig(f'gan_generated_images/image_at_epoch_{epoch:04d}.png')
    plt.show()

In [ ]:
def train(dataset, epochs):

    for epoch in range(1, epochs + 1):

        for image_batch in dataset:
            train_step(image_batch)

        display.clear_output(wait=True)

        generate_and_save_images(generator, epoch, seed)

        print(f'Epoch {epoch}/{epochs} completed')

        # Сохраняем модели каждые 10 эпох
        if epoch % 10 == 0:
            generator.save(
                f'gan_saved_models/generator_epoch_{epoch}.keras'
            )

            discriminator.save(
                f'gan_saved_models/discriminator_epoch_{epoch}.keras'
            )

In [ ]:
train(train_dataset, EPOCHS)

### **Демонстрация сгенерированных изображений:**

In [ ]:
noise = tf.random.normal([16, noise_dim])

generated_images = generator(noise, training=False)

plt.figure(figsize=(4, 4))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow((generated_images[i, :, :, 0] + 1) / 2.0, cmap='gray')
    plt.axis('off')

plt.show()

---

## **Задание №2.** Обучите генератор воспризводить примитивные изображения по заданному условию (Conditional Generative Adversarial Nets (CGAN)).



(На вход генератора подается вектор случайного шума и метка класса - на выходе должно получиться изображение, соответствующее данному классу)



Датасет выберите по желанию. ([Пример №1](https://www.kaggle.com/datasets/andrewmvd/medical-mnist), [Пример №2](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/fashion_mnist/load_data#example), [Пример №3](https://www.kaggle.com/datasets/sagyamthapa/handwritten-math-symbols))

In [ ]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
os.makedirs("cgan_generated_images", exist_ok=True)
os.makedirs("cgan_saved_models", exist_ok=True)

In [ ]:
(x_train, y_train), (_, _) = tf.keras.datasets.fashion_mnist.load_data()

# Нормализация
x_train = (x_train.astype("float32") - 127.5) / 127.5

# Добавляем канал
x_train = np.expand_dims(x_train, axis=-1)

print("Images shape:", x_train.shape)
print("Labels shape:", y_train.shape)

Что ожидать Images shape: (60000, 28, 28, 1)
Labels shape: (60000,)

In [ ]:
BUFFER_SIZE = 60000
BATCH_SIZE = 256
EPOCHS = 50

noise_dim = 100
num_classes = 10

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)

train_dataset = (
    train_dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
)

In [ ]:
def build_generator():

    noise_input = layers.Input(shape=(noise_dim,))
    label_input = layers.Input(shape=(1,))

    # Embedding класса
    label_embedding = layers.Embedding(num_classes, 50)(label_input)
    label_embedding = layers.Flatten()(label_embedding)

    # Объединяем шум и embedding
    model_input = layers.Concatenate()(
        [noise_input, label_embedding]
    )

    x = layers.Dense(7 * 7 * 256, use_bias=False)(model_input)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    x = layers.Reshape((7, 7, 256))(x)

    x = layers.Conv2DTranspose(
        128,
        5,
        strides=1,
        padding='same',
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2DTranspose(
        64,
        5,
        strides=2,
        padding='same',
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    output = layers.Conv2DTranspose(
        1,
        5,
        strides=2,
        padding='same',
        activation='tanh'
    )(x)

    model = tf.keras.Model(
        [noise_input, label_input],
        output
    )

    return model

In [ ]:
def build_discriminator():

    image_input = layers.Input(shape=(28, 28, 1))
    label_input = layers.Input(shape=(1,))

    # Embedding класса
    label_embedding = layers.Embedding(num_classes, 50)(label_input)
    label_embedding = layers.Dense(28 * 28)(label_embedding)

    label_embedding = layers.Reshape((28, 28, 1))(
        label_embedding
    )

    # Объединяем изображение и embedding
    x = layers.Concatenate(axis=-1)(
        [image_input, label_embedding]
    )

    x = layers.Conv2D(
        64,
        5,
        strides=2,
        padding='same'
    )(x)

    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(
        128,
        5,
        strides=2,
        padding='same'
    )(x)

    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Flatten()(x)

    output = layers.Dense(1)(x)

    model = tf.keras.Model(
        [image_input, label_input],
        output
    )

    return model

In [ ]:
generator = build_generator()
discriminator = build_discriminator()

generator.summary()
discriminator.summary()

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(
    from_logits=True
)

def generator_loss(fake_output):
    return cross_entropy(
        tf.ones_like(fake_output),
        fake_output
    )

def discriminator_loss(real_output, fake_output):

    real_loss = cross_entropy(
        tf.ones_like(real_output),
        real_output
    )

    fake_loss = cross_entropy(
        tf.zeros_like(fake_output),
        fake_output
    )

    return real_loss + fake_loss

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

In [ ]:
@tf.function
def train_step(images, labels):

    batch_size = tf.shape(images)[0]

    noise = tf.random.normal([batch_size, noise_dim])

    random_labels = tf.random.uniform(
        [batch_size, 1],
        minval=0,
        maxval=num_classes,
        dtype=tf.int32
    )

    labels = tf.cast(
        tf.reshape(labels, (-1, 1)),
        tf.int32
    )

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:

        generated_images = generator(
            [noise, random_labels],
            training=True
        )

        real_output = discriminator(
            [images, labels],
            training=True
        )

        fake_output = discriminator(
            [generated_images, random_labels],
            training=True
        )

        gen_loss = generator_loss(fake_output)

        disc_loss = discriminator_loss(
            real_output,
            fake_output
        )

    gradients_of_generator = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    gradients_of_discriminator = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    generator_optimizer.apply_gradients(
        zip(
            gradients_of_generator,
            generator.trainable_variables
        )
    )

    discriminator_optimizer.apply_gradients(
        zip(
            gradients_of_discriminator,
            discriminator.trainable_variables
        )
    )

    return gen_loss, disc_loss

In [ ]:
class_names = [
    "T-shirt",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

In [ ]:
def generate_images_by_class(model, class_label):

    noise = tf.random.normal([16, noise_dim])

    labels = tf.constant(
        [[class_label]] * 16,
        dtype=tf.int32
    )

    generated_images = model(
        [noise, labels],
        training=False
    )

    plt.figure(figsize=(4, 4))

    for i in range(16):

        plt.subplot(4, 4, i + 1)

        plt.imshow(
            (generated_images[i, :, :, 0] + 1) / 2.0,
            cmap='gray'
        )

        plt.axis('off')

    plt.suptitle(
        f"Generated class: {class_names[class_label]}"
    )

    plt.show()

In [ ]:
def train(dataset, epochs):

    for epoch in range(epochs):

        for image_batch, label_batch in dataset:

            gen_loss, disc_loss = train_step(
                image_batch,
                label_batch
            )

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Generator Loss: {gen_loss:.4f} | "
            f"Discriminator Loss: {disc_loss:.4f}"
        )

        # Сохраняем модели каждые 10 эпох
        if (epoch + 1) % 10 == 0:

            generator.save(
                f"cgan_saved_models/generator_epoch_{epoch+1}.keras"
            )

            discriminator.save(
                f"cgan_saved_models/discriminator_epoch_{epoch+1}.keras"
            )

In [ ]:
train(train_dataset, EPOCHS)

In [ ]:
generate_images_by_class(generator, 7)

In [ ]:
generate_images_by_class(generator, 8)

In [ ]:
generate_images_by_class(generator, 3)

### **Демонстрация сгенерированных изображений:**

In [ ]:
# Ваш код

---

## **Задание №3.** Обучите генератор колоризировать изображения из выбранного Вами датасета (можете использовать датасет из работы №6, в которой Вы решали аналогичную задачу).

In [ ]:
# Ваш код

### **Демонстрация сгенерированных изображений:**

In [ ]:
# Ваш код

---

## **Задание №4.** Обучите генератор воспроизводить изображения из выбранного Вами датасета (pix2pix).



---
Примеры таких датасетов представлены на сайтах [kaggle.com](https://www.kaggle.com/search?q=pix2pix+in%3Adatasets+datasetFileTypes%3Ajpg+datasetFileTypes%3Apng) и [efrosgans.eecs.berkeley.edu*](http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/).

**\*Калифорнийский университет Беркли (UC Berkeley) внесён в реестр организаций, деятельность которых признана нежелательной в России.**


---

In [ ]:
# Ваш код

### **Демонстрация сгенерированных изображений:**

In [ ]:
# Ваш код